## Overall Process for SNOMED-GPS Processing for OCL:

1. Process the current year's text file to put it into OCL Bulk Import format
2. Identify terms that are in the current year's file but not in the previous year's file, which will be appended to the current year's file but with "retired": true
3. Review numbers of new vs. retired concepts before proceeding
4. Create a json lines-formatted file (but with a .json extension)

## Configuration

### Subtask:
Define the file paths for the current year's raw SNOMED-GPS data, the previous year's OCL concepts file (JSON Lines format), and the OpenRefine-style processing rules file. This centralizes file path management for easier adjustments.


**Reasoning**:
To centralize file path management, I will define variables for the current year's raw data file, the previous year's OCL concepts file, and the processing rules file, assigning them the specified string values.



In [82]:
raw_data_file_path = '/content/drive/MyDrive/OCL Shared/Content Publication/SNOMED-GPS/SNOMED-GPS 2025/SnomedINTL_GPSRelease_PRODUCTION_20250701T120000Z.txt'
previous_year_file_path = '/content/drive/MyDrive/OCL Shared/Content Publication/SNOMED-GPS/Legacy Artifacts/SNOMED-GPS-for-OCL-2024.json'
processing_rules_file_path = '/content/drive/MyDrive/OCL Shared/Content Publication/SNOMED-GPS/SNOMED-GPS 2025/Step 4 - ProcessingSteps.json'

output_file_path = '/content/drive/MyDrive/OCL Shared/Content Publication/SNOMED-GPS/SNOMED-GPS 2025/2025 SNOMED-GPS Import-test.json'

print(f"Raw data file path: {raw_data_file_path}")
print(f"Previous year concepts file path: {previous_year_file_path}")
print(f"Processing rules file path: {processing_rules_file_path}")

Raw data file path: /content/drive/MyDrive/OCL Shared/Content Publication/SNOMED-GPS/SNOMED-GPS 2025/SnomedINTL_GPSRelease_PRODUCTION_20250701T120000Z.txt
Previous year concepts file path: /content/drive/MyDrive/OCL Shared/Content Publication/SNOMED-GPS/Legacy Artifacts/SNOMED-GPS-for-OCL-2024.json
Processing rules file path: /content/drive/MyDrive/OCL Shared/Content Publication/SNOMED-GPS/SNOMED-GPS 2025/Step 4 - ProcessingSteps.json


## Load Processing Rules and Previous Year Concepts

### Subtask:
Load the processing rules from `Step 4 - ProcessingSteps.json` into a dictionary. Concurrently, load the previous year's concepts from `SNOMED-GPS-for-OCL-2024.json`, ensuring correct parsing for its JSON Lines format. Implement robust error handling for `FileNotFoundError` and `json.JSONDecodeError`.


**Reasoning**:
To load the JSON files, I need to import the `json` module and then open and load each file into the specified variables with appropriate error handling, as described in the instructions.



In [83]:
import json

previous_year_concepts = [] # Initialize as empty list in case file is not found

try:
    # Load the previous year's concepts (JSON Lines format)
    with open(previous_year_file_path, 'r', encoding='utf-8') as f:
        for line in f:
            previous_year_concepts.append(json.loads(line))
    print("Previous year's concepts loaded successfully.")
except FileNotFoundError:
    print(f"Warning: Previous year's concepts file '{previous_year_file_path}' not found. It will be treated as an empty list.")
except json.JSONDecodeError as e:
    print(f"Error decoding JSON from previous year's concepts file: {e}. Ensure it is a valid JSON Lines file.")

processing_rules = {} # Initialize as empty dictionary in case file is not found

try:
    # Load the processing rules
    with open(processing_rules_file_path, 'r', encoding='utf-8') as f:
        processing_rules = json.load(f)
    print("Processing rules loaded successfully.")
except FileNotFoundError:
    print(f"Warning: Processing rules file '{processing_rules_file_path}' not found. It will be treated as an empty dictionary.")
except json.JSONDecodeError as e:
    print(f"Error decoding JSON from processing rules file: {e}. Ensure it is a valid JSON file.")

if previous_year_concepts and processing_rules:
    print("Both previous year's concepts and processing rules are available.")
elif previous_year_concepts:
    print("Only previous year's concepts are available.")
elif processing_rules:
    print("Only processing rules are available.")
else:
    print("Neither previous year's concepts nor processing rules could be loaded.")

Previous year's concepts loaded successfully.
Processing rules loaded successfully.
Both previous year's concepts and processing rules are available.


## Process Current Year's Raw Data

### Subtask:
Read and parse the raw SNOMED-GPS text file. Apply the processing rules from 'Step 4 - ProcessingSteps.json' to convert this raw text data into a structured list of dictionaries, representing the current year's SNOMED-GPS concepts in OCL Bulk Import format.


**Reasoning**:
The next step is to read the raw SNOMED-GPS data, parse it based on an assumed tab delimiter, and then apply the loaded processing rules to transform each concept into the OCL Bulk Import format. This involves renaming columns and adding new columns with fixed values as specified in the `processing_rules`.



In [84]:
current_year_concepts_processed = []

# Define the concept ID to check, for debugging purposes
debug_concept_id = "1240411000000107"

# Assuming tab-separated values, as no explicit delimiter was found in processing_rules
delimiter = '\t'

try:
    with open(raw_data_file_path, 'r', encoding='utf-8') as f:
        header_line = f.readline().strip()
        raw_headers = header_line.split(delimiter)

        for line in f:
            values = line.strip().split(delimiter)

            # Skip empty lines or lines with incorrect number of values
            if not line.strip() or len(values) != len(raw_headers):
                continue

            concept_dict = dict(zip(raw_headers, values))

            # Check if the debug_concept_id is in the raw concept_dict (assuming 'ConceptID' is the original ID column)
            if 'ConceptID' in concept_dict and concept_dict['ConceptID'] == debug_concept_id:
                print(f"DEBUG: Found raw concept ID {debug_concept_id} before processing: {concept_dict}")

            processed_concept = concept_dict.copy()

            # Apply processing rules
            for rule in processing_rules:
                operation = rule.get('op')

                if operation == 'core/column-rename':
                    old_name = rule.get('oldColumnName')
                    new_name = rule.get('newColumnName')
                    if old_name in processed_concept:
                        processed_concept[new_name] = processed_concept.pop(old_name)

                elif operation == 'core/column-addition':
                    new_col_name = rule.get('newColumnName')
                    expression = rule.get('expression')

                    # Handle GREL expressions for fixed values
                    if expression and expression.startswith('grel:"') and expression.endswith('"'):
                        value_to_add = expression[len('grel:"'):-1]
                        processed_concept[new_col_name] = value_to_add
                    elif rule.get('baseColumnName') and rule.get('baseColumnName') in processed_concept:
                        # If it's an addition based on an existing column, but no expression, copy the value
                        processed_concept[new_col_name] = processed_concept[rule.get('baseColumnName')]
                    else:
                        # Default for other column additions if no specific value or base column
                        processed_concept[new_col_name] = ""

                # More complex rules like 'core/value-transform', 'core/row-filter' would be implemented here if needed.

            # Check if the debug_concept_id is in the processed_concept dictionary after rules (assuming 'id' is the final ID column)
            if 'id' in processed_concept and processed_concept['id'] == debug_concept_id:
                print(f"DEBUG: Found processed concept ID {debug_concept_id} after processing rules: {processed_concept}")

            current_year_concepts_processed.append(processed_concept)

    print(f"Successfully processed {len(current_year_concepts_processed)} concepts from the raw data file.")
except FileNotFoundError:
    print(f"Warning: Raw data file '{raw_data_file_path}' not found. No concepts processed.")
except Exception as e:
    print(f"An error occurred during raw data processing: {e}")

DEBUG: Found raw concept ID 1240411000000107 before processing: {'ConceptID': '1240411000000107', 'Active': '1', 'FSN': 'Ribonucleic acid of severe acute respiratory syndrome coronavirus 2 (substance)', 'USPreferredTerm': 'Severe acute respiratory syndrome coronavirus 2 RNA'}
DEBUG: Found processed concept ID 1240411000000107 after processing rules: {'Active': '1', 'FSN': 'Ribonucleic acid of severe acute respiratory syndrome coronavirus 2 (substance)', 'USPreferredTerm': 'Severe acute respiratory syndrome coronavirus 2 RNA', 'id': '1240411000000107', 'type': 'Concept', 'owner': 'SNOMED-International', 'owner_type': 'Organization', 'retired': '', 'datatype': 'None', 'concept_class': 'Misc', 'source': 'SNOMED-GPS'}
Successfully processed 29577 concepts from the raw data file.


## Identify New and Retired Concepts

### Subtask:
Compare the processed current year's concepts with the loaded previous year's concepts, using only the 'id' field as the determinant. Identify concepts that are exclusively in the current year's data (new concepts) and those exclusively in the previous year's data (retired concepts). Include print statements to show the total unique IDs from both years to aid in debugging if counts are unexpected.


**Reasoning**:
To identify new and retired concepts, I will extract the unique 'id' from both the current and previous year's concept lists, convert them into sets for efficient comparison, then determine the set differences to find new and retired concept IDs. Finally, I will filter the original concept lists to gather the full dictionaries for these identified new and retired concepts and print the requested counts.



In [85]:
current_concept_ids = {c['id'] for c in current_year_concepts_processed if 'id' in c}
previous_concept_ids = {c['id'] for c in previous_year_concepts if 'id' in c}

print(f"Total unique IDs in current year's data: {len(current_concept_ids)}")
print(f"Total unique IDs in previous year's data: {len(previous_concept_ids)}")

# Identify new concept IDs (in current but not in previous)
new_concept_ids = current_concept_ids - previous_concept_ids

# Identify retired concept IDs (in previous but not in current)
retired_concept_ids = previous_concept_ids - current_concept_ids

# Get the full concept dictionaries for new concepts
new_concepts = [c for c in current_year_concepts_processed if c['id'] in new_concept_ids]

# Get the full concept dictionaries for retired concepts
retired_concepts = [c for c in previous_year_concepts if c['id'] in retired_concept_ids]

print(f"Identified {len(new_concepts)} new concepts.")
# print("New Concept IDs:")
# for concept_id in new_concept_ids:
#     print(f"- {concept_id}")

print(f"Identified {len(retired_concepts)} retired concepts.")
# print("Retired Concept IDs:")
# for concept_id in retired_concept_ids:
#     print(f"- {concept_id}")

Total unique IDs in current year's data: 29577
Total unique IDs in previous year's data: 28994
Identified 1211 new concepts.
Identified 628 retired concepts.


In [86]:
# Check if a specific concept ID exists in either set
concept_id_to_check = "1240411000000107"

print(f"\n--- Checking Concept ID: {concept_id_to_check} ---")

if concept_id_to_check in current_concept_ids:
    print(f"'{concept_id_to_check}' is present in the current year's concepts.")
else:
    print(f"'{concept_id_to_check}' is NOT present in the current year's concepts.")

if concept_id_to_check in previous_concept_ids:
    print(f"'{concept_id_to_check}' is present in the previous year's concepts.")
else:
    print(f"'{concept_id_to_check}' is NOT present in the previous year's concepts.")


--- Checking Concept ID: 1240411000000107 ---
'1240411000000107' is present in the current year's concepts.
'1240411000000107' is present in the previous year's concepts.


## Prepare Final OCL Import Data

### Subtask:
Combine the accurately processed current year's concepts with the identified retired concepts. For every retired concept, add a "retired": true attribute before adding it to the final dataset for OCL bulk import. This step ensures the combined dataset is ready for export, with appropriate flags for retired items.


**Reasoning**:
To combine the current year's processed concepts with the identified retired concepts, I will initialize an empty list, add all current concepts, then iterate through the retired concepts, marking each with 'retired': true before adding them to the combined list, and finally print the total count.



In [87]:
final_ocl_import_data = []

# Add current year's processed concepts, explicitly setting 'retired' to False
for concept in current_year_concepts_processed:
    active_concept = concept.copy()
    active_concept['retired'] = False
    final_ocl_import_data.append(active_concept)

# Process and add retired concepts
for concept in retired_concepts:
    retired_concept = concept.copy() # Create a copy to avoid modifying the original list if it's referenced elsewhere
    retired_concept['retired'] = True
    final_ocl_import_data.append(retired_concept)

print(f"Total concepts for OCL Bulk Import: {len(final_ocl_import_data)} (Current: {len(current_year_concepts_processed)}, Retired marked: {len(retired_concepts)})")

Total concepts for OCL Bulk Import: 30205 (Current: 29577, Retired marked: 628)


## Review Concept Changes

### Subtask:
Display the final calculated counts for new and retired concepts based on the comparison. This step serves as a critical review point to verify the accuracy of identified changes before proceeding to file export.


**Reasoning**:
To display the calculated counts for new and retired concepts, I will use a code block to print the length of the `new_concepts` and `retired_concepts` lists, along with a descriptive message as requested.



In [88]:
print(f"Total new concepts: {len(new_concepts)}")
print(f"Total retired concepts: {len(retired_concepts)}")
print("These numbers represent the changes identified for review before finalizing the import file.")

Total new concepts: 1211
Total retired concepts: 628
These numbers represent the changes identified for review before finalizing the import file.


## Export Final JSON Lines File

### Subtask:
Save the complete and prepared OCL import data (current year's concepts plus retired concepts with the `retired` flag) into a new file, `2025 SNOMED-GPS Import.json`, ensuring it is formatted as JSON Lines. This provides the final output ready for OCL import.


**Reasoning**:
To save the combined OCL import data in JSON Lines format, I will define the output file path, then iterate through the `final_ocl_import_data` list, converting each concept to a JSON string and writing it as a line in the specified output file, ensuring proper JSON Lines format, and include error handling.



In [89]:
columns_to_drop = []

for concept in final_ocl_import_data:
    names_list = []

    # Transform FSN into 'Fully-Specified' name
    if "FSN" in concept and concept["FSN"]:
        names_list.append({
            "locale": "en",
            "locale_preferred": False,
            "name": concept["FSN"],
            "name_type": "Fully-Specified"
        })

    # Transform USPreferredTerm into 'Synonym' name
    if "USPreferredTerm" in concept and concept["USPreferredTerm"]:
        names_list.append({
            "locale": "en",
            "locale_preferred": True,
            "name": concept["USPreferredTerm"],
            "name_type": "Synonym"
        })

    if names_list:
        concept["names"] = names_list

    # Drop the original FSN and USPreferredTerm columns, along with 'Active'
    for col in ["FSN", "USPreferredTerm"] + columns_to_drop:
        if col in concept:
            del concept[col]

print(f"Transformed 'FSN' and 'USPreferredTerm' into 'names' list and dropped columns {["FSN", "USPreferredTerm"] + columns_to_drop} from all concepts.")

Transformed 'FSN' and 'USPreferredTerm' into 'names' list and dropped columns ['FSN', 'USPreferredTerm'] from all concepts.


In [90]:
try:
    with open(output_file_path, 'w', encoding='utf-8') as outfile:
        for concept in final_ocl_import_data:
            json.dump(concept, outfile)
            outfile.write('\n') # Ensure JSON Lines format
    print(f"Successfully exported {len(final_ocl_import_data)} concepts to '{output_file_path}' in JSON Lines format.")
except Exception as e:
    print(f"Error exporting OCL bulk import data: {e}")

Successfully exported 30205 concepts to '/content/drive/MyDrive/OCL Shared/Content Publication/SNOMED-GPS/SNOMED-GPS 2025/2025 SNOMED-GPS Import-test.json' in JSON Lines format.
